# Creador de vídeos AI: MoneyPrinterTurbo completo en Google Colab

Este notebook levanta la WebUI y API de MoneyPrinterTurbo, el buscador ScrapeGraphAI con NVIDIA NIM, y el flujo de render Qwen-Image + Remotion. El código se ejecuta en `/content`; clips, trabajos y renders se guardan en Google Drive. Colab necesita una sesión con GPU para Qwen.

**Preparación:** en el panel **Secrets** de Colab, crea `NVIDIA_API_KEY` y `NVIDIA_MODEL` y permite que este notebook los lea. Como una clave anterior apareció en texto de la conversación, revócala y guarda una nueva en Secrets; no la escribas en una celda ni la guardes en Drive. Opcionalmente agrega `YOUTUBE_API_KEY`, `INSTAGRAM_ACCESS_TOKEN`, `INSTAGRAM_USER_ID`, `TIKTOK_RESEARCH_CLIENT_KEY` y `TIKTOK_RESEARCH_CLIENT_SECRET` para las APIs oficiales que tengas aprobadas. El modelo NVIDIA debe admitir chat completions OpenAI-compatible.

La aplicación queda disponible mediante el proxy autenticado de Colab, no mediante un túnel público. La GPU se usa para inferencia Qwen; Remotion/FFmpeg consume principalmente CPU y RAM. Qwen-Image-2.1 y Remotion tienen licencias propias; revísalas antes de uso comercial.

In [ ]:
from google.colab import drive, output, userdata
from pathlib import Path
import os, re, shutil, subprocess, sys, time, urllib.request

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/MoneyPrinterTurbo')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
REPO_DIR = Path('/content/Creador-videos-Ai')
INTEGRATION_DIR = REPO_DIR / 'moneyprinter-colab'
MPT_DIR = Path('/content/MoneyPrinterTurbo')
MPT_COMMIT = '405ece42c4167038c6cf58e6f79d7bc0ff75da93'
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/javiertarazon/Creador-videos-Ai.git', str(REPO_DIR)], check=True)
if not (INTEGRATION_DIR / 'colab' / 'colab_runner.py').is_file():
    raise FileNotFoundError(f'No está publicada la integración: {INTEGRATION_DIR}')
if not (MPT_DIR / 'config.example.toml').is_file():
    if MPT_DIR.exists() and any(MPT_DIR.iterdir()):
        raise RuntimeError(f'La carpeta {MPT_DIR} existe pero no parece un clon limpio de MoneyPrinterTurbo.')
    MPT_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'init', str(MPT_DIR)], check=True, stdout=subprocess.DEVNULL)
    subprocess.run(['git', '-C', str(MPT_DIR), 'remote', 'add', 'origin', 'https://github.com/javiertarazon/MoneyPrinterTurbo.git'], check=True)
    subprocess.run(['git', '-C', str(MPT_DIR), 'fetch', '--depth', '1', 'origin', MPT_COMMIT], check=True)
    subprocess.run(['git', '-C', str(MPT_DIR), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
if not (MPT_DIR / 'config.example.toml').is_file():
    raise FileNotFoundError(f'El clon de MoneyPrinterTurbo no contiene config.example.toml: {MPT_DIR}')

def required_secret(name):
    try:
        value = userdata.get(name)
    except Exception as exc:
        raise RuntimeError(f'Crea {name} en el panel Secrets de Colab y habilita su acceso para este notebook.') from exc
    if not value or not value.strip():
        raise RuntimeError(f'El secreto {name} está vacío. Configúralo en Secrets de Colab.')
    return value.strip()

def optional_secret(name, default=''):
    try:
        value = userdata.get(name)
        return value.strip() if value else default
    except Exception:
        return default

NVIDIA_API_KEY = required_secret('NVIDIA_API_KEY')
NVIDIA_MODEL = required_secret('NVIDIA_MODEL')
YOUTUBE_API_KEY = optional_secret('YOUTUBE_API_KEY')
INSTAGRAM_ACCESS_TOKEN = optional_secret('INSTAGRAM_ACCESS_TOKEN')
INSTAGRAM_USER_ID = optional_secret('INSTAGRAM_USER_ID')
TIKTOK_RESEARCH_CLIENT_KEY = optional_secret('TIKTOK_RESEARCH_CLIENT_KEY')
TIKTOK_RESEARCH_CLIENT_SECRET = optional_secret('TIKTOK_RESEARCH_CLIENT_SECRET')

ORIGINALS_DIR = DRIVE_ROOT / 'videos-originales'
DATA_DIR = DRIVE_ROOT / 'datos'
REPORTS_DIR = DRIVE_ROOT / 'informes-tendencias'
PLANS_DIR = DRIVE_ROOT / 'planes-contenido'
INBOX_DIR = DRIVE_ROOT / 'remotion-inbox'
RENDERS_DIR = DRIVE_ROOT / 'renders'
for directory in (ORIGINALS_DIR, DATA_DIR, REPORTS_DIR, PLANS_DIR, INBOX_DIR, RENDERS_DIR):
    directory.mkdir(parents=True, exist_ok=True)
storage_link = MPT_DIR / 'storage'
persistent_storage = DATA_DIR / 'storage'
persistent_storage.mkdir(parents=True, exist_ok=True)
if storage_link.is_symlink():
    if storage_link.resolve() != persistent_storage.resolve():
        raise RuntimeError(f'La ruta storage ya apunta a otra ubicación: {storage_link.resolve()}')
elif storage_link.exists():
    if not storage_link.is_dir() or any(storage_link.iterdir()):
        raise RuntimeError(f'No se reemplaza el storage existente. Revisa manualmente {storage_link}.')
    storage_link.rmdir()
if not storage_link.exists():
    storage_link.symlink_to(persistent_storage, target_is_directory=True)
print('Código:', INTEGRATION_DIR)
print('MoneyPrinterTurbo:', MPT_DIR)
print('Clips propios en Drive:', ORIGINALS_DIR)
print('No se muestran ni imprimen las claves cargadas desde Secrets.')

## Instalar dependencias y preparar el runtime

Se usan entornos virtuales separados para MoneyPrinterTurbo y ScrapeGraphAI, evitando que sus dependencias de Python se interfieran. El runtime es temporal; al iniciar una nueva sesión vuelve a ejecutar las celdas.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Activa GPU en Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU y vuelve a ejecutar.')
print('GPU:', torch.cuda.get_device_name(0), '| VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GiB')

subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'nodejs', 'npm'], check=True)
MPT_VENV = Path('/content/venvs/mpt')
SCRAPER_VENV = Path('/content/venvs/scraper')
for venv in (MPT_VENV, SCRAPER_VENV):
    if not (venv / 'bin' / 'python').is_file():
        subprocess.run([sys.executable, '-m', 'venv', str(venv)], check=True)
subprocess.run([str(MPT_VENV / 'bin' / 'pip'), 'install', '-r', str(MPT_DIR / 'requirements.txt')], check=True)
subprocess.run([str(SCRAPER_VENV / 'bin' / 'pip'), 'install', '-r', str(INTEGRATION_DIR / 'scraper' / 'requirements.txt')], check=True)
subprocess.run([str(SCRAPER_VENV / 'bin' / 'playwright'), 'install', '--with-deps', 'chromium'], check=True)

for filename in ('design_profiles.py', 'remotion_export.py', 'scraper_client.py', 'trend_search.py'):
    shutil.copy2(INTEGRATION_DIR / filename, MPT_DIR / filename)
shutil.copytree(INTEGRATION_DIR / 'webui-pages', MPT_DIR / 'webui' / 'pages', dirs_exist_ok=True)

config_template = (MPT_DIR / 'config.example.toml').read_text(encoding='utf-8')
def set_toml_value(text, name, value):
    pattern = re.compile(rf'(?m)^{re.escape(name)}\s*=.*$')
    replacement = f'{name} = {__import__("json").dumps(value, ensure_ascii=False)}'
    updated, count = pattern.subn(replacement, text, count=1)
    if count == 0 and name in {'youtube_api_key', 'instagram_access_token', 'instagram_user_id', 'instagram_graph_version', 'tiktok_research_client_key', 'tiktok_research_client_secret'}:
        section = re.search(r'(?m)^\[app\]\s*$', text)
        if not section:
            raise RuntimeError('No se encontró la sección [app] en config.example.toml.')
        following = re.search(r'(?m)^\[.+\]\s*$', text[section.end():])
        insert_at = section.end() + following.start() if following else len(text)
        return text[:insert_at] + replacement + '\n' + text[insert_at:]
    if count != 1:
        raise RuntimeError(f'No se encontró la opción {name} en config.example.toml.')
    return updated

for key, value in {
    'listen_host': '127.0.0.1',
    'llm_provider': 'openai',
    'openai_api_key': NVIDIA_API_KEY,
    'openai_base_url': 'https://integrate.api.nvidia.com/v1',
    'openai_model_name': NVIDIA_MODEL,
    'video_source': 'local',
    'youtube_api_key': YOUTUBE_API_KEY,
    'instagram_access_token': INSTAGRAM_ACCESS_TOKEN,
    'instagram_user_id': INSTAGRAM_USER_ID,
    'instagram_graph_version': 'v25.0',
    'tiktok_research_client_key': TIKTOK_RESEARCH_CLIENT_KEY,
    'tiktok_research_client_secret': TIKTOK_RESEARCH_CLIENT_SECRET,
}.items():
    config_template = set_toml_value(config_template, key, value)
(MPT_DIR / 'config.toml').write_text(config_template, encoding='utf-8')
NVIDIA_CONFIG = Path('/content/nvidia-scraper.toml')
import json
NVIDIA_CONFIG.write_text('[nvidia]\n' + '\n'.join(f'{key} = {json.dumps(value)}' for key, value in {
    'base_url': 'https://integrate.api.nvidia.com/v1',
    'api_key': NVIDIA_API_KEY,
    'model': NVIDIA_MODEL,
}.items()) + '\n', encoding='utf-8')
os.chmod(NVIDIA_CONFIG, 0o600)

print('Dependencias instaladas; configuración NVIDIA creada solo en el disco temporal de Colab.')

## Iniciar la aplicación completa

La WebUI se sirve por el proxy de Colab. Los procesos de API y scraper escuchan solo en loopback; no se crea un túnel público. Sube tus propios clips a `MoneyPrinterTurbo/videos-originales` en Drive antes de seleccionarlos en **Idea a video**.

In [ ]:
import json
env = os.environ.copy()
env.update({
    'MPT_API_URL': 'http://127.0.0.1:8080',
    'MPT_CONFIG_PATH': str(MPT_DIR / 'config.toml'),
    'MPT_ORIGINALS_DIR': str(ORIGINALS_DIR),
    'MPT_REPORTS_DIR': str(REPORTS_DIR),
    'MPT_PLANS_DIR': str(PLANS_DIR),
    'MPT_DATA_DIR': str(DATA_DIR),
    'SOCIAL_SCRAPER_URL': 'http://127.0.0.1:8090',
    'NVIDIA_SCRAPER_CONFIG': str(NVIDIA_CONFIG),
    'PYTHONPATH': str(MPT_DIR),
})
LOG_DIR = Path('/content/logs')
LOG_DIR.mkdir(exist_ok=True)
SERVICES = {}
def service_responds(url):
    try:
        with urllib.request.urlopen(url, timeout=2) as response:
            return response.status < 500
    except Exception:
        return False

def start_service(name, command, cwd, health_url, service_env):
    if service_responds(health_url):
        print(name, 'ya responde')
        return
    log = (LOG_DIR / f'{name}.log').open('w', encoding='utf-8')
    process = subprocess.Popen(command, cwd=cwd, env=service_env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    SERVICES[name] = (process, log)
    deadline = time.time() + 180
    while time.time() < deadline:
        if service_responds(health_url):
            print(name, 'saludable (PID', process.pid, ')')
            return
        if process.poll() is not None:
            raise RuntimeError(f'{name} terminó con código {process.returncode}; revisa {LOG_DIR / (name + ".log")}')
        time.sleep(2)
    raise TimeoutError(f'{name} no respondió a tiempo; revisa {LOG_DIR / (name + ".log")}')

scraper_env = env.copy()
scraper_env.pop('PYTHONPATH', None)
start_service('scraper', [str(SCRAPER_VENV / 'bin' / 'python'), '-m', 'uvicorn', 'service:app', '--host', '127.0.0.1', '--port', '8090'], INTEGRATION_DIR / 'scraper', 'http://127.0.0.1:8090/health', scraper_env)
start_service('api', [str(MPT_VENV / 'bin' / 'python'), 'main.py'], MPT_DIR, 'http://127.0.0.1:8080/docs', env)
start_service('webui', [str(MPT_VENV / 'bin' / 'streamlit'), 'run', './webui/Main.py', '--server.address=0.0.0.0', '--server.port=8501', '--server.enableCORS=false', '--browser.gatherUsageStats=false'], MPT_DIR, 'http://127.0.0.1:8501/_stcore/health', env)
print('Abre la WebUI:')
output.serve_kernel_port_as_window(8501)
print('Documentación del API (opcional):')
output.serve_kernel_port_as_window(8080)

## Generar el vídeo con Qwen + Remotion

1. En la WebUI abre **Idea a video**, busca una idea, genera y revisa el guion, selecciona tus clips y pulsa **Preparar paquete para Colab**.
2. Guarda el ZIP descargado en `Mi unidad/MoneyPrinterTurbo/remotion-inbox/`.
3. Ejecuta la celda siguiente. Toma el ZIP más reciente y guarda el MP4, imágenes y manifiestos en `Mi unidad/MoneyPrinterTurbo/renders/`.

In [ ]:
%pip install --upgrade 'transformers>=5.17' accelerate pillow 'git+https://github.com/huggingface/diffusers'
import torch
from IPython.display import Video, display
sys.path.insert(0, str(INTEGRATION_DIR / 'colab'))
from colab_runner import IMAGE_MODEL_ID, render_bundle
sys.path.insert(0, str(INTEGRATION_DIR))
if not torch.cuda.is_available():
    raise RuntimeError('La sesión ya no tiene GPU. Actívala en la configuración del runtime y vuelve a ejecutar.')
bundles = sorted(INBOX_DIR.glob('*.zip'), key=lambda path: path.stat().st_mtime_ns)
if not bundles:
    raise FileNotFoundError(f'Sube el paquete ZIP a {INBOX_DIR}')
rendered_video = render_bundle(
    bundle_path=bundles[-1],
    project_template=INTEGRATION_DIR / 'remotion',
    output_dir=RENDERS_DIR,
    generate_images=True,
    model_id=IMAGE_MODEL_ID,
)
print('Vídeo guardado en Drive:', rendered_video)
display(Video(str(rendered_video), embed=False))